# Hung matched Development — shard 0001
Exactly 20 jobs. Reuses immutable Source/Target dependencies; pilot_0000 is excluded. Private Save & Run All output requires a completed version and independent download/SHA/CRC verification. No Final or peer approval claim.

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, shutil, subprocess, sys, time
from datetime import datetime, timezone
base=Path('/kaggle/working/hung-matched-shard0001-v1')
assert not base.exists(), 'Preserve existing output; do not run this version twice'
payloads=list(Path('/kaggle/input').rglob('provider-payload.zip.bin'))
assert len(payloads)==1, 'Attach exact private payload dataset'
payload=payloads[0]
helper=payload.parent/'hung_provider.py'
assert hashlib.sha256(helper.read_bytes()).hexdigest()=='cadcd80d671d3b49a317312f125735362ffba3068247f1714a0ae56c6be83424'
assert hashlib.sha256(payload.read_bytes()).hexdigest()=='23a6d420523d990d07ff07d814ed642a851925283a102a3ae806f3b666e1d6f4'
spec=importlib.util.spec_from_file_location('hung_provider',helper)
provider=importlib.util.module_from_spec(spec)
spec.loader.exec_module(provider)
repo,plan_path=provider.materialize(payload,'23a6d420523d990d07ff07d814ed642a851925283a102a3ae806f3b666e1d6f4',base)
os.chdir(repo)
sys.path.insert(0,str(repo))
import torch
assert torch.cuda.is_available(), 'CUDA GPU required'
assert shutil.disk_usage(base).free >= 8*1024**3, '8 GiB free required'
environment={'torch':torch.__version__,'python':sys.version,'cuda':torch.version.cuda,
 'device':torch.cuda.get_device_name(0),'memory_bytes':torch.cuda.get_device_properties(0).total_memory,
 'disk_free_bytes':shutil.disk_usage(base).free,'started_at_utc':datetime.now(timezone.utc).isoformat(),
 'quota_observed_before_launch':'23h57m available of 30h, owner UI; subsequent usage must be rechecked',
 'code_sha':'44b8532959d76fb20691881539f991ca44983727','shard':'shard_0001',
 'scope':'Development execution authorized by user; independent acceptance and Final remain pending'}
provider.save(base/'environment.json',environment)
print(json.dumps(environment))


In [ ]:
"""Bounded CPU/CUDA portability check on two permitted initial Development examples."""
import json
from copy import deepcopy
import time
from pathlib import Path

import torch

from experiments.hung_sprint2 import dispatch_matched as dispatch
from experiments.hung_sprint2.cli import _examples, _tensors
from experiments.hung_sprint2.dispatch_state import atomic_create
from experiments.hung_sprint2.model import ForecastGRU, ModelConfig
from experiments.hung_sprint2.retention import retention_loss

torch.set_num_threads(1)
ATOL, RTOL = 1e-5, 1e-4
root = Path.cwd()
plan_path = root / '.cache/provider/plan.json'
plan = dispatch.read_verified_json(plan_path,dispatch.sha256_file(plan_path))
job = next(row for row in plan['jobs'] if row['id']=='online_best_b14s7e1153')
cell = next(row for row in plan['cells'] if row['id']==job['cell'])
config_pin = next(row for row in plan['configs'] if row['id']==job['config'])
config = dispatch.read_verified_json(Path(config_pin['path']),config_pin['sha256'])
input_run=Path(plan['input_base'])/'urbanev-SZH-14d'
manifest=dispatch._load_run(input_run,cell['input_manifest_sha256'])
rows,count=_examples(input_run,manifest,'development',cell['entity_id'],limit=2)
assert rows and torch.cuda.is_available()
x,c,y=_tensors(rows)
states={}
for name in ('source','target'):
    dispatch.verify_file(Path(job[name]['path']),job[name]['sha256'])
    states[name]=torch.load(job[name]['path'],map_location='cpu',weights_only=True)

def evaluate(device):
    target=ForecastGRU(ModelConfig(**states['target']['architecture'])).to(device).train()
    source=ForecastGRU(ModelConfig(**states['source']['architecture'])).to(device).eval()
    target.load_state_dict(states['target']['model_state'])
    source.load_state_dict(states['source']['model_state'])
    source.requires_grad_(False)
    optimizer=torch.optim.Adam(target.parameters(),lr=config['target']['learning_rate'])
    optimizer.load_state_dict(deepcopy(states['target']['optimizer_state']))
    prediction,latent=target.forecast_and_latent(x.to(device),c.to(device))
    with torch.no_grad():
        anchor=source.encode(x.to(device))
    forecast=(prediction-y.to(device)).abs().mean()
    retain=retention_loss(latent,anchor,states['source']['latent_scale'].to(device),epsilon=config['target']['epsilon'])
    loss=forecast+config['target']['retention_weight']*retain
    loss.backward()
    result={'forecast':prediction.detach().cpu(),'latent':latent.detach().cpu(),
            'forecast_loss':forecast.detach().cpu(),'retention_loss':retain.detach().cpu(),
            **{'gradient/'+name:parameter.grad.detach().cpu().clone() for name,parameter in target.named_parameters()}}
    optimizer.step()
    result.update({'updated/'+name:parameter.detach().cpu().clone() for name,parameter in target.named_parameters()})
    return result

started=time.perf_counter()
cpu,gpu=evaluate('cpu'),evaluate('cuda')
checks={key:{'max_absolute_difference':float((cpu[key]-gpu[key]).abs().max()),
             'within_tolerance':bool(torch.allclose(cpu[key],gpu[key],atol=ATOL,rtol=RTOL))}
        for key in cpu}
passed=all(row['within_tolerance'] for row in checks.values())
report={'schema':'hung-local-cuda-bounded-parity-v1','passed':passed,'atol':ATOL,'rtol':RTOL,
        'job_id':job['id'],'plan_sha256':dispatch.sha256_file(plan_path),'code':dispatch._code_identity(),
        'input_manifest_sha256':cell['input_manifest_sha256'],'source':job['source'],'target':job['target'],
        'permitted_initial_examples_used':len(rows),'permitted_initial_examples_available':count,
        'elapsed_seconds':time.perf_counter()-started,'device':torch.cuda.get_device_name(0),'torch':torch.__version__,
        'checks':checks,'scope':'Initial Development forward/loss/gradient/one-Adam-update portability only; no saved model update or research result'}
atomic_create(root.parent/'provider-parity.json',report)
print(json.dumps({'passed':passed,'checks':len(checks),'elapsed_seconds':report['elapsed_seconds']}))
assert passed, 'Provider parity failed; do not execute shard'


In [ ]:
assert passed, 'Numerical parity prerequisite'
t0=time.perf_counter()
command=[sys.executable,'-u','-m','experiments.hung_sprint2.dispatch_matched',
 '--plan',str(plan_path),'--plan-sha256',provider.sha256(plan_path),
 '--shards',str(repo/'.cache/provider/shards.json'),'--shards-sha256',provider.sha256(repo/'.cache/provider/shards.json'),
 '--shard-id','shard_0001']
with (base/'dryrun.log').open('x') as log:
 result=subprocess.run(command,cwd=repo,stdout=log,stderr=subprocess.STDOUT)
provider.save(base/'dryrun-exit.json',{'exit_code':result.returncode,'wall_seconds':time.perf_counter()-t0})
assert result.returncode==0, 'Dry-run failed; do not train'
receipt=provider.execute_shard(repo,'shard_0001',base/'shard_0001')
print(json.dumps(receipt))
print('LOCAL_GPU_EXECUTION_COMPLETE_REMOTE_READBACK_PENDING')
